# NB6 · Whole-body allocation and supervision

Use the spatial extension of the canonical four-bar with four joint moments.
Compare weighted QP and hierarchical QP on one compliant simulation plant.
This is a thin-disk research model for horizontal support surfaces with synthetic transverse inertia,
contact, driver bounds and sensor channels. The allocator uses individual normal
and longitudinal constraints plus a common lateral rolling row. The plant
retains separate per-wheel forces and allows slip and separation.

[DUK v3 chapter 08](https://app.notion.com/p/4131f3cc03328156af1dc6b76714fce1)
and [chapter 09](https://app.notion.com/p/4131f3cc0332811da657f1bbb15cb66b).

In [ ]:
import hashlib
import json
import subprocess
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from duk_core.modeling import load_leg_geometry_json
from duk_core.visualization.notebook import configure_notebook_plots

configure_notebook_plots()

repo = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "robot/config/physical/duk_leg_geometry.json").is_file()
)
geometry_path = repo / "robot/config/physical/duk_leg_geometry.json"
fixture_path = repo / "research/experiments/duk-v3/synthetic.json"
geometry = load_leg_geometry_json(geometry_path)
fixture = json.loads(fixture_path.read_text())
assert fixture["schema_version"] == 1 and fixture["status"] == "synthetic_research_only"
print(
    "Working tree dirty:",
    bool(subprocess.check_output(["git", "status", "--porcelain"], cwd=repo)),
)
print(
    "Commit",
    subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip(),
)
for path in (geometry_path, fixture_path):
    print(path.relative_to(repo), hashlib.sha256(path.read_bytes()).hexdigest())
print(fixture["source"])


configure_notebook_plots()

In [ ]:
from collections import Counter

from duk_core.modeling import build_planar_biped_model
from duk_core.modeling.hip_load import LinearTorqueSpeed
from duk_core.modeling.planar_numeric import NumericPlanarBiped
from duk_core.modeling.spatial_biped import SpatialBiped
from duk_core.simulation.whole_body import run_whole_body

planar = NumericPlanarBiped(build_planar_biped_model(geometry), fixture["planar"])
s = fixture["spatial"]
plant = SpatialBiped(planar, s["lateral_mounts_m"], s["transverse_inertias_kg_m2"])
p = fixture["whole_body"]
env = fixture["hip_envelope"]
motor = LinearTorqueSpeed(
    env["stall_torque_N_m"], env["no_load_speed_rad_s"], env["efficiency"]
)
q = np.r_[0.08, 0.45, 0.13, np.deg2rad([94.0, 103.0]), 0.12, -0.22]
v = np.array([0.12, -0.04, 0.2, 0.3, -0.2, -0.7, 0.6])
state, tangent = plant.sagittal_state(q, v)
d = plant.evaluate(state)
flat = planar.evaluate(q, v)
np.testing.assert_allclose(tangent.T @ d.mass @ tangent, flat.mass, atol=1e-10)
np.testing.assert_allclose(tangent.T @ d.actuation, flat.input, atol=1e-10)
np.testing.assert_allclose(
    d.kinematics.contact[:, 0] @ tangent, flat.contact[2:], atol=1e-10
)
np.testing.assert_allclose(
    d.kinematics.contact[:, 2] @ tangent, flat.contact[:2], atol=1e-10
)
assert np.linalg.eigvalsh(d.mass)[0] > 0.0
print("Spatial mass, actuation and contact agree in the sagittal reduction.")


def run(name, method="weighted", priority="balance_first"):
    scenario = p["scenario"] | p["cases"][name]
    candidate = plant
    if "support_levels_m" in scenario:
        candidate = SpatialBiped(
            planar,
            s["lateral_mounts_m"],
            s["transverse_inertias_kg_m2"],
            support_levels_m=scenario["support_levels_m"],
        )
    return run_whole_body(
        candidate,
        motor,
        fixture["v2_balance"],
        p,
        scenario,
        method=method,
        priority=priority,
    )


results = {name: run(name) for name in p["cases"]}
results["competing_hqp_balance"] = run("competing", "hierarchical", "balance_first")
results["competing_hqp_legs"] = run("competing", "hierarchical", "legs_first")
for name, trace in results.items():
    assert np.all(abs(trace.applied_effort) <= trace.effort_limit + 1e-8)
    assert np.all(trace.force[:, :, 2] >= 0.0)
    solved = np.array(trace.solver_status) == "solved"
    assert np.all(trace.solver_residual[solved] <= 2e-6)
    assert np.isfinite(trace.pose).all()
    assert trace.friction_belief.shape == (len(trace.time), 2)
    assert np.isfinite(trace.friction_belief).all() and np.all(
        trace.friction_belief >= 0
    )
    peak = np.rad2deg(trace.peak_abs_lean_roll)
    reached_horizon = trace.stop_reason is None and np.isclose(
        trace.terminal_time, (p["scenario"] | p["cases"].get(name, {}))["duration_s"]
    )
    performance = (
        reached_horizon
        and peak[0] <= p["maximum_lean_deg"]
        and peak[1] <= p["maximum_roll_deg"]
    )
    print(
        name,
        "completed with declared angles:",
        performance,
        "peak lean/roll (deg):",
        peak.round(3),
        "solver states:",
        dict(Counter(trace.solver_status)),
        "stop:",
        trace.stop_reason,
        "max solve ms:",
        round(1000 * trace.solver_seconds.max(), 2),
    )

In [ ]:
nominal = results["nominal"]
assert len(nominal.time) == round(p["scenario"]["duration_s"] / p["period_s"])
assert nominal.terminal_time == p["scenario"]["duration_s"]
assert nominal.stop_reason is None and all(
    status == "solved" for status in nominal.solver_status
)
assert np.rad2deg(nominal.peak_abs_lean_roll[0]) <= p["maximum_lean_deg"]
assert np.rad2deg(nominal.peak_abs_lean_roll[1]) <= p["maximum_roll_deg"]
assert np.max(abs(nominal.pose[:, 3])) > 0.005
assert "Active" in [status[0] for status in nominal.feature_status]

off = results["drive_release"]
assert not off.feature_intent[:, 1:].any()
release = off.time >= p["cases"]["drive_release"]["drive_end_s"]
assert np.all(off.effective_reference[release, 2] == 0.0)

slip = results["low_grip"]
np.testing.assert_allclose(
    slip.friction_belief[0], p["scenario"]["friction"] * p["friction_reserve"]
)
assert np.any(slip.friction_belief[-1] < slip.friction_belief[0])
changed = slip.time >= p["cases"]["low_grip"]["friction_change_s"]
np.testing.assert_allclose(slip.actual_friction[~changed], p["scenario"]["friction"])
np.testing.assert_allclose(
    slip.actual_friction[changed], p["cases"]["low_grip"]["friction_after"]
)

sliding = np.array(["sliding" in modes for modes in slip.contact_modes])
assert sliding.any() and np.all(slip.feature_amount[sliding] == 0.0)
assert "Paused" in [status[2] for status in slip.feature_status]

late, infeasible = results["late"], results["infeasible"]
for trace, fault_status in ((late, "deadline"), (infeasible, "infeasible")):
    rejected = np.array(trace.solver_status) == fault_status
    assert rejected.any()
    # Reduced mode can already apply balance_fallback. If its validity guard
    # fails at a later tick, a zero-command stop is the required response.
    sources = np.array(trace.command_source)
    assert np.isin(sources[rejected], ("balance_fallback", "stop")).all()
    assert np.all(trace.feature_amount[rejected] == 0.0)
    if trace.stop_reason is not None:
        assert trace.supervisor[-1] == "stop" and sources[-1] == "stop"
        assert np.all(trace.applied_effort[-1] == 0.0)
brief = results["brief_late"]
assert "deadline" in brief.solver_status and brief.supervisor[-1] == "normal"
sensor = results["sensor_lost"]
assert sensor.stop_reason == "sensor" and np.all(sensor.applied_effort[-1] == 0.0)
assert (
    sensor.solver_status[-1] == "skipped_sensor" and sensor.command_source[-1] == "stop"
)
fallback = np.array(brief.command_source) == "balance_fallback"
assert fallback.any() and np.all(brief.requested_reference[fallback, 2] > 0.0)
assert np.all(brief.effective_reference[fallback, 1:] == 0.0)
assert np.all(brief.feature_amount[fallback] == 0.0)
air = results["extended_flight"]
assert ("flight", "flight") in air.contact_modes and air.stop_reason == "flight_or_fall"
brief_air = results["brief_flight"]
assert ("flight", "flight") in brief_air.contact_modes and brief_air.contact_modes[
    -1
] == ("rolling", "rolling")
one_wheel = results["one_wheel"]
assert any(modes.count("flight") == 1 for modes in one_wheel.contact_modes)
for key in ("brief_flight", "one_wheel"):
    event = results[key]
    print(key, "observed contact sequence:", tuple(dict.fromkeys(event.contact_modes)))
levels = results["split_level"]
assert abs(levels.pose[0, 2]) > 0.02
assert abs(levels.terminal_pose[2]) < abs(levels.pose[0, 2])
print(
    "Split-level roll initial/final (deg):",
    np.rad2deg([levels.pose[0, 2], levels.terminal_pose[2]]).round(3),
)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 6), constrained_layout=True)
axes[0, 0].plot(nominal.time, nominal.requested_reference[:, 0], label="Requested")
axes[0, 0].plot(nominal.time, nominal.effective_reference[:, 0], label="Admitted")
axes[0, 0].plot(nominal.time, nominal.pose[:, 1], label="Actual")
axes[0, 0].set_ylabel("Body height (m)")
axes[0, 0].legend()
axes[0, 1].plot(nominal.time, nominal.requested_reference[:, 1], label="Requested")
axes[0, 1].plot(nominal.time, nominal.effective_reference[:, 1], label="Admitted")
axes[0, 1].plot(nominal.time, nominal.pose_rate[:, 3], label="Actual")
axes[0, 1].set_ylabel("Yaw rate (rad/s)")
axes[0, 1].legend()
axes[0, 2].plot(slip.position_xy[:, 0], slip.position_xy[:, 1], label="Low grip")
axes[0, 2].plot(nominal.position_xy[:, 0], nominal.position_xy[:, 1], label="Nominal")
axes[0, 2].set(xlabel="x (m)", ylabel="y (m)")
axes[0, 2].legend()
for key, label in [
    ("competing", "Weighted"),
    ("competing_hqp_balance", "HQP balance first"),
    ("competing_hqp_legs", "HQP legs first"),
]:
    trace = results[key]
    axes[1, 0].plot(trace.time, np.rad2deg(trace.pose[:, 0]), label=label)
axes[1, 0].set_ylabel("Equivalent lean (deg)")
axes[1, 0].legend()
axes[1, 1].plot(slip.time, slip.force[:, :, 2], label=["Left", "Right"])
axes[1, 1].set_ylabel("Normal force (N)")
axes[1, 1].legend()
axes[1, 2].plot(
    slip.time, slip.feature_amount, label=["Bounce", "Level", "Anchor", "Heading"]
)
axes[1, 2].set_ylabel("Feature participation")
axes[1, 2].legend()
for ax in [*axes[1], *axes[0, :2]]:
    ax.set_xlabel("Time (s)")
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(10, 5), constrained_layout=True)
for i, axis in enumerate(axes.flat):
    axis.plot(late.time, late.requested_effort[:, i], "--", label="Candidate/request")
    axis.plot(late.time, late.applied_effort[:, i], label="Applied")
    axis.set(
        xlabel="Time (s)", ylabel=("Hip L", "Hip R", "Wheel L", "Wheel R")[i] + " (N·m)"
    )
    axis.legend()
plt.show()
fig, axes = plt.subplots(1, 2, figsize=(9, 3), constrained_layout=True)
axes[0].plot(levels.time, np.rad2deg(levels.pose[:, 2]), label="Body roll")
axes[0].axhline(0.0, linestyle="--", color="k", label="Level reference")
axes[0].set(xlabel="Time (s)", ylabel="Split-level roll (deg)")
axes[1].plot(levels.time, levels.force[:, :, 2], label=["Left", "Right"])
axes[1].set(xlabel="Time (s)", ylabel="Split-level normal force (N)")
for ax in axes:
    ax.legend()
plt.show()

**Scope of the comparison.** The supervisor consumes simulated pose and an
independently compensated acceleration channel. Contact observations and
applied forces come from the simulation. Low-grip and fault cases expose the
consequences of the selected research policy, including stopped trials. The
headroom governor uses measured command utilization from the preceding sample.
Hierarchy and solver times are evaluated on this Python process. Physical
sensor, drivetrain, road and stop-policy validation require separately
identified data.

Controller rows record the state before each held command. `terminal_pose`
records the final integrated state. Peak lean/roll includes every integration
substep, where fall and hip travel guards are checked. The QP bounds cover
the entire constant-acceleration hip prediction, including an interior turning
point. These remain checks of the numerical prediction and sampled plant.
Rolling odometry uses the tilted disk support geometry and compensated body,
carrier and wheel rates without using world translation.

**Corrected chapter 04 scope (2026-11-11).** The 10-coordinate 3D model has
five independent rolling rows only when the wheel axles are collinear
(equal hip posture in this model): two normals, two longitudinal rows and
one independent lateral row. Unequal hip posture can give rank six. The
allocator's common lateral row is an approximation outside that aligned
case; the compliant plant still keeps both per-wheel tangent directions.

The chapter 08 condition A(2πf)² < g assumes the height signal is approximately
the **CoM** height and is only a necessary support condition. This notebook
commands body-reference height; its actual unilateral forces are evaluated
by the plant rather than imposing that approximate formula as an exact limit.

Bounce feedforward includes the first derivative of feature participation and its cross term with the sine wave. Participation retains the existing piecewise-linear ramp. Its rate can jump at ramp starts, target changes, saturation or pause; the acceleration supplied is the regular derivative between those breakpoints, not an impulse or a claim of a C² reference.

A stress trial can stop before the scheduled feature-off time. Feature Exit behavior is checked directly in the policy tests; completion of a particular stress trial is not required to reach that transition. The effort, contact, angle and stop checks above remain independent of that schedule.


## Outcome dashboard

The angle criterion requires completion and both declared angle limits. A stopped
fault trial remains visible with its actual end time and stop reason. Peak values
include every integration substep; terminal markers show the final integrated
state. Timelines end at that time. Select a cached trial without rerunning it.

Impulse times follow the sampled experiment clock: the first command tick at or
after the configured instant receives the impulse. Friction and fault changes
use the same sample-boundary convention.

Command and feature curves use a zero-order hold through the actual terminal
time, including a partial last interval. A stop logged exactly at that time
remains an instantaneous change, with no invented interval after the stop.

In [ ]:
from IPython.display import HTML, display

from duk_core.visualization.notebook import (
    case_selector,
    plot_held_signal,
    plot_status_timeline,
    result_table,
)

rows = []
for name, trace in results.items():
    horizon = (p["scenario"] | p["cases"].get(name, {}))["duration_s"]
    completed = trace.stop_reason is None and np.isclose(trace.terminal_time, horizon)
    peak = np.rad2deg(trace.peak_abs_lean_roll)
    passed = (
        completed
        and peak[0] <= p["maximum_lean_deg"]
        and peak[1] <= p["maximum_roll_deg"]
    )
    rows.append(
        {
            "Case": name,
            "Run": "COMPLETED" if completed else "STOPPED",
            "Angle criterion": "PASS" if passed else "FAIL",
            "End (s)": trace.terminal_time,
            "Peak lean (deg)": peak[0],
            "Peak roll (deg)": peak[1],
            "Solved (%)": 100 * np.mean(np.array(trace.solver_status) == "solved"),
            "Stop reason": trace.stop_reason or "—",
        }
    )
display(
    HTML(
        result_table(
            rows,
            caption="NB6 · synthetic scenarios · lean/roll limits "
            f"{p['maximum_lean_deg']:g}° / {p['maximum_roll_deg']:g}°",
        )
    )
)


def draw_whole_body_case(name):
    trace = results[name]
    fig, axes = plt.subplots(
        3, 2, figsize=(12, 9), sharex=True, constrained_layout=True
    )
    for column, label, limit in (
        (0, "Lean", p["maximum_lean_deg"]),
        (2, "Roll", p["maximum_roll_deg"]),
    ):
        (line,) = axes[0, 0].plot(
            trace.time, np.rad2deg(trace.pose[:, column]), label=label
        )
        axes[0, 0].plot(
            trace.terminal_time,
            np.rad2deg(trace.terminal_pose[column]),
            "o",
            color=line.get_color(),
            clip_on=False,
        )
        for sign in (-1, 1):
            axes[0, 0].axhline(sign * limit, color=line.get_color(), ls=":", alpha=0.6)
    axes[0, 0].set(ylabel="Angle (deg)", title="Body angles · dots mark terminal state")
    for i, label in enumerate(("Hip L", "Hip R", "Wheel L", "Wheel R")):
        (line,) = plot_held_signal(
            axes[0, 1],
            trace.time,
            trace.applied_effort[:, i],
            end_time=trace.terminal_time,
            label=label,
        )
        plot_held_signal(
            axes[0, 1],
            trace.time,
            trace.effort_limit[:, i],
            end_time=trace.terminal_time,
            color=line.get_color(),
            ls=":",
            alpha=0.5,
        )
        plot_held_signal(
            axes[0, 1],
            trace.time,
            -trace.effort_limit[:, i],
            end_time=trace.terminal_time,
            color=line.get_color(),
            ls=":",
            alpha=0.5,
        )
    axes[0, 1].set(ylabel="Effort (N·m)", title="Applied effort · dotted bounds")
    plot_status_timeline(
        axes[1, 0],
        trace.time,
        trace.supervisor,
        end_time=trace.terminal_time,
        title="Supervisor",
    )
    plot_status_timeline(
        axes[1, 1],
        trace.time,
        trace.solver_status,
        end_time=trace.terminal_time,
        title="Allocator status",
    )
    modes = [f"L: {left} / R: {right}" for left, right in trace.contact_modes]
    plot_status_timeline(
        axes[2, 0],
        trace.time,
        modes,
        end_time=trace.terminal_time,
        title="Observed wheel contact modes",
    )
    plot_held_signal(
        axes[2, 1],
        trace.time,
        trace.feature_amount,
        end_time=trace.terminal_time,
        label=["Bounce", "Level", "Anchor", "Heading"],
    )
    axes[2, 1].set(
        ylabel="Participation",
        ylim=(-0.05, 1.05),
        title="Admitted feature contribution",
    )
    for ax in axes.flat:
        ax.set_xlabel("Time (s)")
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize=8)
    for ax in axes[:-1].flat:
        ax.set_xlabel("")
    reason = trace.stop_reason or "completed"
    fig.suptitle(f"{name} · {reason} · terminal time {trace.terminal_time:.3f} s")
    return fig


whole_body_picker = case_selector(tuple(results), draw_whole_body_case)

## Friction observations and belief

The plant friction coefficient is a synthetic scenario input. The belief updates
from tangential/normal force ratios during observed sliding with enough normal
force. The ratio uses the Euclidean tangent norm, consistent with the isotropic
plant law and independent of the direction of slip. The allocator retains its
inscribed linear friction constraints.

Force observations are from the preceding integration substep. A change of
surface friction can precede its observation; the belief trace exposes this delay.


In [ ]:
grip = results["low_grip"]
fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
plot_held_signal(
    axes[0],
    grip.time,
    grip.actual_friction[:, 0],
    end_time=grip.terminal_time,
    color="0.2",
    linewidth=2,
    label="Surface μ (simulated)",
)
plot_held_signal(
    axes[0],
    grip.time,
    grip.friction_belief,
    end_time=grip.terminal_time,
    label=["Left belief", "Right belief"],
)
axes[0].set(
    ylabel="Friction coefficient",
    title="Low-grip scenario · plant input and controller belief",
)
normal = grip.force[:, :, 2]
force_ratio = np.divide(
    np.linalg.norm(grip.force[:, :, :2], axis=2),
    normal,
    out=np.full_like(normal, np.nan),
    where=normal > p["minimum_force_N"],
)
axes[1].plot(grip.time, force_ratio, label=["Left observation", "Right observation"])
axes[1].set(
    xlabel="Time (s)", ylabel="‖Fₜ‖₂ / N", title="Observed contact-force ratios"
)
for ax in axes:
    ax.legend(fontsize=8)
plt.show()

## Conditional contact rank

These are local algebraic ranks of the full six contact-velocity rows
with a level body, not a declaration that either pose is in a valid
rigid-support contact mode. Five remaining velocity directions in the
aligned case do not imply five independently actuated coordinates.

In [ ]:
rank_rows = []
for angles in ((95.0, 95.0), (90.0, 110.0)):
    probe, _ = plant.sagittal_state(
        np.r_[0.0, 0.4, 0.0, np.deg2rad(angles), 0.0, 0.0], np.zeros(7)
    )
    rank = np.linalg.matrix_rank(plant.kinematics(probe).contact.reshape(6, 10))
    rank_rows.append(
        {
            "Hip L (deg)": angles[0],
            "Hip R (deg)": angles[1],
            "Full contact rank": rank,
            "Nullity": 10 - rank,
        }
    )
assert [row["Full contact rank"] for row in rank_rows] == [5, 6]
display(
    HTML(
        result_table(
            rank_rows,
            caption="Full 3D contact rows · conditional rank · four actuators",
        )
    )
)